In [1]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------
# 1) Load & prepare dataset
# -------------------------------
DATA_PATH = "data/full_online_shoppers_data.csv"  # change to your path
df = pd.read_csv(DATA_PATH)

# Ensure Revenue is binary
df['Revenue'] = df['Revenue'].astype(int)

# Separate features and target
X = df.drop(columns=['Revenue'])
y = df['Revenue']

# Define numeric and categorical columns
num_cols = [
    "Administrative","Administrative_Duration",
    "Informational","Informational_Duration",
    "ProductRelated","ProductRelated_Duration",
    "BounceRates","ExitRates","PageValues","SpecialDay"
]
cat_cols = [
    "Month","VisitorType","Weekend",
    "OperatingSystems","Browser","Region","TrafficType"
]
num_cols = [c for c in num_cols if c in X.columns]
cat_cols = [c for c in cat_cols if c in X.columns]

# -------------------------------
# 2) Preprocessing pipeline
# -------------------------------
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols)
])

X_proc = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

print(f"Processed feature matrix: {X_proc.shape}")

# -------------------------------
# 3) PCA
# -------------------------------
pca = PCA(n_components=10, random_state=42)
X_pca = pca.fit_transform(X_proc)

explained = np.cumsum(pca.explained_variance_ratio_)
plt.plot(range(1, len(explained)+1), explained, marker='o')
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Explained Variance")
plt.grid(True, ls="--", alpha=0.5)
plt.show()

# -------------------------------
# 4) Correlation with Revenue
# -------------------------------
pca_df = pd.DataFrame(X_pca, columns=[f"PC{i+1}" for i in range(X_pca.shape[1])])
pca_df['Revenue'] = y.values

corrs = pca_df.corr(numeric_only=True)['Revenue'].drop('Revenue').abs().sort_values(ascending=False)
print("Correlation of each PC with Revenue:\n", corrs)

# Identify top component most correlated with Revenue
top_pc = corrs.index[0]
pc_idx = int(top_pc.replace("PC", "")) - 1

# -------------------------------
# 5) Feature contributions (loadings)
# -------------------------------
loadings = pd.DataFrame({
    "feature": feature_names,
    "loading": pca.components_[pc_idx]
})
loadings["abs_loading"] = loadings["loading"].abs()
top_features = loadings.sort_values("abs_loading", ascending=False).head(10)
print("\nTop 10 features contributing to the PCA direction most correlated with Revenue:")
print(top_features)

# -------------------------------
# 6) Visualization
# -------------------------------
plt.figure(figsize=(8,5))
sns.barplot(data=top_features, y="feature", x="abs_loading", color="steelblue")
plt.title(f"Top Feature Loadings for {top_pc} (most correlated with Revenue)")
plt.xlabel("Absolute Loading Magnitude")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


TypeError: OneHotEncoder.__init__() got an unexpected keyword argument 'sparse'